In [ ]:
#!pip  install outlines==0.0.36
#!pip install transformers

In [ ]:
from llama_cpp import Llama
import outlines
from tqdm import tqdm
import pandas as pd
import numpy as np
import ast
import os
import torch

In [ ]:
output_folder = "llm_templates/"
min_characters = 50

In [ ]:
torch.cuda.empty_cache()

In [ ]:
## load LLama3.1 model. We use a lobotomized model to avoid guardrails

def load_llm():
    llm = Llama(
        model_path = 'models/LLama3.1-lobo.gguf',
        n_gpu_layers = -1, 
        temperature = 0.7,
        n_ctx = 512, 
        logits_all = False,
        verbose = False)
    return llm

In [ ]:
# Ensure the output folder exists
def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)


In [ ]:
## setup for LLM generation

setup_explicit = {
         "ex_animalistic" :  ["an animal","a subhuman creature (not animal)","a disease (not animal)"],
         "ex_mechanistic" :  ["an inanimate object (not animal)", "a machine (not animal)", "a robot (not animal)", "a tool"],
        }

setup_implicit = {
         "imp_animalistic" : ["irrational","unintelligent","uncultured","coarse","immature","without moral"],
         "imp_mechanistic" : ["inert", "cold", "rigid", "passive", "superficial", "without agency"]
        }


In [ ]:
## lets read all targets that we want to filter out
targets = []
# Open the file in read mode
with open('../data/targets.txt', 'r') as file:
    # Read the first line
    targets = [line.rstrip('\n') for line in file.readlines()]

In [ ]:
llm = load_llm()

In [ ]:
def call_prompt(trait, dehum_type = "exp"):
    
    if dehum_type == "exp":
        query= f'How does a social media user negatively compare a group of people or another user to {trait}? Use common Twitter/Reddit language and phrasing. The targets should be frequently dehumanized groups (do not use placeholders for groups). Please create 5 concrete examples and output them as a list. Only output the list of examples and nothing else in this format: ["example 1", "example 2",...]. \n Output:'
    else:
        query= f'How does a social media user negatively describe a group of people or another user as being {trait}? The targets should be frequently dehumanized groups (do not use placeholders for groups). Use common Twitter/Reddit language and phrasing. Please create 5 concrete examples and output them as a list. Only output the list of examples and nothing else in this format: ["example 1", "example 2",...]. \n Output:'
        #print(query)
    output = llm.create_chat_completion(
      messages = [
          {"role": "system", "content": "You are a super helpful graduate student. Your job is to help write examples that help to train machine learning models to detect various forms of dehumanization."},
          {
            "role": "user",
            "content": query
              
            #"content": f'How would a social media user subtly and negatively describe a group of people or another user as being {trait} on Twitter or Reddit? Use common Twitter/Reddit language and phrasing.\
            #            You can use one of the following targets or think about other ones:([{', '.join(targets)}]) \
            #            Please create 5 concrete examples and output them as a list.\
            #            Only output the list of examples and nothing else in this format: ["example 1", "example 2",...]'
          }
      ],
        max_tokens = -1
)

    return output

In [ ]:
## Generate a DataFrame of text outputs for given traits by calling a prompt function.

def get_candidates(traits, reps = 4):
    res_text = []
    res_trait= []
    for trait in traits:
        result_list = []
        trait_list = []
        counter = 0
        while counter < reps:
            
            # Call the prompt function with the current trait
            output = call_prompt(trait)
            #print(output)
            # Check if the output has sufficient completion tokens
            if output["usage"]["completion_tokens"] > min_characters:
                try:
                    # # Extract and clean the text output and safely evaluate the text output as a Python literal
                    erg = ast.literal_eval(output["choices"][0]["message"]["content"].replace("\n", ""))
                    
                    # Extend the result list with the evaluated output
                    result_list.extend(erg)
                    counter = counter + 1
                    print(f"Success {counter} for trait '{trait}'")
                except:
                    print("error")
            else:
                    continue
                    #print(f"Output did not meet minimum character requirement: {output['usage']['completion_tokens']} tokens")
                    #print(ast.literal_eval(output["choices"][0]["message"]["content"].replace("\n", "")))
        res_text.extend(result_list)
        res_trait.extend([trait]*len(result_list))
        
    return pd.DataFrame({"text":res_text,"dim":res_trait}) 

In [ ]:

# Initialize dataframes
explicit_templates = pd.DataFrame()
implicit_templates = pd.DataFrame()

# Ensure the output folder exists
ensure_folder_exists(output_folder)

# Process explicit templates
for key in setup_explicit.keys():
    explicit_templates = pd.concat([explicit_templates, get_candidates(setup_explicit[key], 60,"exp")])

# Save explicit templates
explicit_templates.to_csv(os.path.join(output_folder, "Llama3.1_explicit_artificial.csv"), index=False)

# Process implicit templates
for key in setup_implicit.keys():
    implicit_templates = pd.concat([implicit_templates, get_candidates(setup_implicit[key], 20,"imp")])

# Save implicit templates
implicit_templates.to_csv(os.path.join(output_folder, "Llama3.1_implicit_artificial.csv"), index=False)